In [1]:
!pip install -q pyngrok diffusers transformers accelerate fastapi uvicorn pydantic pillow

In [ ]:
import base64
import gc
import io
import os
import numpy as np
from PIL import Image, ImageOps
import torch
from diffusers import (
    AutoencoderKL,
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    StableDiffusionImg2ImgPipeline,
    UniPCMultistepScheduler,
)
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn

# 1. Initialize FastAPI Application
app = FastAPI(title="Colab Blueprint-to-Isometric 3D Engine")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 2. Precision VAE (Eliminates gray haze and improves lighting/color fidelity)
print("Loading Precision FT-MSE VAE...")
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

# 3. Load ControlNet Scribble Model
print("Loading ControlNet Scribble...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-scribble",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

# 4. Realistic Vision V5.1 Pipeline with VAE
print("Loading Realistic Vision V5.1 (Cutaway Architectural Engine)...")
txt2img_pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "SG161222/Realistic_Vision_V5.1_noVAE",
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
    safety_checker=None,
    low_cpu_mem_usage=True
).to("cuda")

txt2img_pipe.scheduler = UniPCMultistepScheduler.from_config(txt2img_pipe.scheduler.config)
txt2img_pipe.enable_attention_slicing()

# 5. Shared-Weight Img2Img Pipeline for Iterative Refinements
print("Configuring Img2Img pipeline for state-preserving edits...")
img2img_pipe = StableDiffusionImg2ImgPipeline(
    vae=txt2img_pipe.vae,
    text_encoder=txt2img_pipe.text_encoder,
    tokenizer=txt2img_pipe.tokenizer,
    unet=txt2img_pipe.unet,
    scheduler=txt2img_pipe.scheduler,
    safety_checker=None,
    feature_extractor=None
).to("cuda")
img2img_pipe.enable_attention_slicing()

# 6. Request Schema
class GenerateRequest(BaseModel):
    image_base64: str
    prompt: str = "modern luxury bedroom, wooden flooring, clean architecture"
    negative_prompt: str = "ugly, distorted, blurry, low quality, dark void, pitch black"
    guidance_scale: float = 6.5
    num_inference_steps: int = 25
    strength: float = 0.28
    is_refinement: bool = False

def clean_sketch(image_bytes: bytes) -> Image.Image:
    """Preprocesses canvas input: resolves transparency and ensures white-on-black lines."""
    raw_img = Image.open(io.BytesIO(image_bytes))
    if raw_img.mode in ("RGBA", "LA") or (raw_img.mode == "P" and "transparency" in raw_img.info):
        alpha_img = raw_img.convert("RGBA")
        bg = Image.new("RGBA", alpha_img.size, (255, 255, 255))
        raw_img = Image.alpha_composite(bg, alpha_img).convert("RGB")
    else:
        raw_img = raw_img.convert("RGB")

    raw_img = raw_img.resize((512, 512))
    gray = raw_img.convert("L")
    if np.mean(gray) > 127:
        gray = ImageOps.invert(gray)
    return gray.convert("RGB")

@app.get("/api/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.is_available()}

@app.post("/api/generate")
def generate_or_refine(req: GenerateRequest):
    try:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        data = req.image_base64
        if "," in data:
            data = data.split(",")[1]
        image_bytes = base64.b64decode(data)

        # 🎯 STRICT PATH A: 3D Isometric Architectural Cutaway Markers
        iso_pos = (
            ", (3d top-down architectural floor plan render:1.3), axonometric cutaway view, "
            "photorealistic architectural visualization, wooden parquet floors, soft natural overhead sunlight, "
            "detailed bed linen, clean plastered walls, architectural model, high fidelity, 8k resolution"
        )
        iso_neg = (
            f"{req.negative_prompt}, eye-level photo, first-person view, front-facing close-up wall, "
            "close-up door, doorway in front of camera, black outlines, flat 2D blueprint diagram, "
            "sketch lines, pencil marks, text labels, watermark, distorted, dark void, pitch black"
        )

        with torch.inference_mode():
            if req.is_refinement:
                # --- TRUE IMG2IMG REFINEMENT (Preserves Isometric Room State) ---
                init_photo = Image.open(io.BytesIO(image_bytes)).convert("RGB").resize((512, 512))
                refine_strength = req.strength if req.strength is not None and 0.35 <= req.strength <= 0.48 else 0.40

                refine_prompt = f"{req.prompt}, 3d architectural floor plan cutaway{iso_pos}"
                refine_negative = (
                    f"{iso_neg}, stone wall, rock masonry, altered layout, "
                    "different pillows, structural rebuilding, shifting walls"
                )

                print(f"[Colab] Running Img2Img Isometric Refinement (strength={refine_strength})...")
                result = img2img_pipe(
                    prompt=refine_prompt,
                    negative_prompt=refine_negative,
                    image=init_photo,
                    strength=refine_strength,
                    guidance_scale=req.guidance_scale,
                    num_inference_steps=req.num_inference_steps
                ).images[0]

            else:
                # --- FRESH CONTROLNET GENERATION (Blueprint -> 3D Isometric Model) ---
                control_sketch = clean_sketch(image_bytes)

                full_prompt = f"A 3d isometric cutaway architectural render of a {req.prompt}{iso_pos}"

                print("[Colab] Running ControlNet 3D Isometric Text-to-Image Generation...")
                result = txt2img_pipe(
                    prompt=full_prompt,
                    negative_prompt=iso_neg,
                    image=control_sketch,
                    controlnet_conditioning_scale=0.88,  # Strongly adheres to floor boundaries
                    guidance_scale=req.guidance_scale,
                    num_inference_steps=req.num_inference_steps
                ).images[0]

        buf = io.BytesIO()
        result.save(buf, format="JPEG", quality=92)
        output_b64 = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode("utf-8")

        return {"status": "success", "image_base64": output_b64, "image_url": output_b64}

    except Exception as e:
        print(f"Error during inference: {e}")
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# 7. Configure ngrok and Run Server
NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN", "YOUR_NGROK_AUTH_TOKEN_HERE")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

public_url = ngrok.connect(8000)
print(f"\n==========================================")
print(f"🚀 PUBLIC BRIDGE URL: {public_url.public_url}")
print(f"==========================================\n")

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

Loading Precision FT-MSE VAE...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading ControlNet Scribble...
Loading Realistic Vision V5.1 (Cutaway Architectural Engine)...


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

...

INFO:     74.220.48.202:0 - "POST /api/generate HTTP/1.1" 200 OK


The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['view , photorealistic architectural visualization , wooden parquet floors , soft natural overhead sunlight , detailed bed linen , clean plastered walls , architectural model , high fidelity , 8 k resolution']


[Colab] Running ControlNet 3D Isometric Text-to-Image Generation...


  0%|          | 0/25 [00:00<?, ?it/s]

INFO:     74.220.48.202:0 - "POST /api/generate HTTP/1.1" 200 OK
